# Inspect the MaMMoS magnetic-SANS data products

This notebook reads one generated HDF5 file through `mammos_entity`, keeping each entity's ontology label, unit, description, and numerical value together. Each file contains one affine magnetization model, its orientation distribution, and the corresponding forward-model results.

| HDF5 group | Stored information |
| --- | --- |
| `model` | Affine coefficients $\mathbf m_0$, $\mathbf m_1$, sphere radius $R$, and orientation density $\psi$. |
| `real_space` | Local body-frame coordinates `x`, `y`, `z` and magnetization components `mx`, `my`, `mz` on a Cartesian grid after the sphere cut. |
| `detector` | Shared detector coordinates $(q,\theta)$ and $(q_y,q_z)$, together with the one-dimensional axes. |
| `average_magnetization` | Orientation-averaged real-space magnetization and the combined orientation- and volume-averaged magnetization, evaluated by QMC and, where available, analytically. |

The reciprocal-space results follow the forward chain

$$
\left\langle\widetilde{\boldsymbol\Gamma}\right\rangle_\psi
\xrightarrow{\boldsymbol\Pi_\perp}
\left\langle\widetilde{\mathbf G}\right\rangle_\psi
\xrightarrow{\mathcal C}
I_{\mathrm M}.
$$

These stages are stored in `average_gamma_tensor`, `average_G_tensor`, and `SANS_cross_sections`, respectively, with two parallel evaluation branches:

- **QMC (`qmc`):** The orientation-averaged $\Gamma$ tensor is estimated using a Sobol sequence of rotations weighted by $\psi$. Its transverse projection gives $G$, from which the basic and derived SANS cross sections are computed.
- **Analytical (`analytical`, where available):** Closed-form expressions provide the corresponding reference results. For cases admitting the joint $\mathrm{SO}(3)$-covariant representation, the three radial spectra $\Gamma_\perp(q)$, $\Gamma_\parallel(q)$, and $\Gamma_\times(q)$ are additionally stored in `average_gamma_tensor/analytical/spectra`. They describe the full averaged $\Gamma$ tensor; only $\Gamma_\perp$ and $\Gamma_\times$ enter the projected $G$ tensor and the purely magnetic SANS cross sections.

Both branches store all nine complex tensor components on the same detector points, together with the basic and derived SANS channels, so numerical and analytical results can be compared directly.

The modules `python.case_studies.case_study1` and `python.case_studies.case_study2` generate the model files in `output/data/`. Launch this notebook with `pixi run notebook` from the repository root and select the HDF5 file using `path` below.

In [ ]:
from pathlib import Path

import mammos_entity as me
import matplotlib.pyplot as plt
import numpy as np

# select one of the datasets
path = Path.cwd().parent / "output/data/case_study1_1.h5"
#path = Path.cwd().parent / "output/data/case_study1_2.h5"
#path = Path.cwd().parent / "output/data/case_study1_3.h5"
#path = Path.cwd().parent / "output/data/case_study2_1.h5"
#path = Path.cwd().parent / "output/data/case_study2_2.h5"

# Read data using mammos-entity

The HDF5 hierarchy is the stored representation of nested `EntityCollection` objects. Each collection is saved as a **group**, while each numerical `Entity` is saved as a **dataset** with attributes for its ontology label and IRI, unit, and description. A generic HDF5 viewer therefore displays groups and datasets.

`me.from_hdf5(path)` reconstructs this hierarchy as MaMMoS Python objects: `data` and its subcollections, such as `data["model"]`, are `EntityCollection` instances; individual numerical entries, such as `data["model"]["m0"]`, are `Entity` instances. The loop below prints the reconstructed types. Access an entity's numerical value through `.value`.

In [ ]:
data = me.from_hdf5(path)

for name, item in data:
    print(name)
    print("  ", type(item))

In [ ]:
data

# Inspect the properties of the model

Display the stored affine coefficients `m0` and `m1`, sphere radius `R`, and orientation distribution `psi` that define the selected model.

In [ ]:
for name, item in data["model"]:
    print(name)
    print("  ", data["model"][name])

# Inspect the real-space data

List the local coordinates and magnetization components inside the sphere. The PyVista view below shows the body-frame field with fixed-length arrows, colored by $M_z/|\mathbf M|$.

In [ ]:
for name, item in data["real_space"]:
    print(name, type(item))

In [ ]:
import pyvista as pv

r = data["real_space"]
points = np.column_stack([r[c].value for c in ("x", "y", "z")])
M = np.column_stack([r[c].value for c in ("mx", "my", "mz")])

field = pv.PolyData(points)
field["M"] = M

norm = np.linalg.norm(M, axis=1)
field["Mz/|M|"] = np.divide(
    M[:, 2],
    norm,
    out=np.zeros_like(norm),
    where=norm > 0,
)

arrow = pv.Arrow(start=(-0.5, 0, 0), scale=1.0)

arrows = field.glyph(
    orient="M",
    scale=False,
    factor=0.15 * data["model"]["R"].value,
    geom=arrow,
)
arrows.plot(
    scalars="Mz/|M|",
    cmap="RdBu",
    clim=(-1, 1),
    background="white",
    jupyter_backend="client",
)


# Inspect the detector coordinates

Inspect the radial and angular axes and the corresponding two-dimensional detector coordinates. The arrays `qy` and `qz` locate the stored scattering data in the detector plane.

In [ ]:
for name, item in data["detector"]:
    print(name, type(item))

In [ ]:
q = data["detector"]["q_axis"]
print(q)

In [ ]:
qz = data["detector"]["qz"]
qy = data["detector"]["qy"]

# Inspect the Fourier-space correlation tensor

List the available QMC and analytical branches of `average_gamma_tensor`. Each branch stores the nine complex Cartesian components of the orientation-averaged correlation tensor on the detector grid.

In [ ]:
gamma = data["average_gamma_tensor"]
for name, item in gamma:
    print(name)
    for name1, item1 in gamma[name]:
        print("  ", name1)
        print("    ", type(item1))

# Inspect the analytical spectra

Read and plot the analytical transverse, longitudinal, and chiral spectra, $\Gamma_\perp$, $\Gamma_\parallel$, and $\Gamma_\times$, against the stored radial q axis.

In [ ]:
gamma_spec = gamma["analytical"]["spectra"]
for name, item in gamma_spec:
    print(name, type(item))

In [ ]:
gamma_par = gamma_spec["parallel"]
gamma_perp = gamma_spec["perpendicular"]
gamma_cross = gamma_spec["cross"]
plt.plot(q.value, gamma_perp.value, label=r"$\Gamma_{\perp}$")
plt.plot(q.value, gamma_par.value, label=r"$\Gamma_{\parallel}$")
plt.plot(q.value, gamma_cross.value, label=r"$\Gamma_{\times}$")
plt.grid()
plt.legend()
plt.ylim(-2, 2)
plt.xlabel(r"qR")
plt.ylabel(r"$\Gamma$")
plt.show()

# Inspect and visualize the magnetic SANS data

Inspect the basic and derived SANS channels, then display the analytical spin-flip intensity $I_{\mathrm{sf}}$ and chiral contribution $I_\chi$ in the detector plane. Repeating the first angular row closes the periodic circle for plotting.

In [ ]:
magSANS_cs = data["SANS_cross_sections"]
for name, item in magSANS_cs:
    print(name)
    for name1, item1 in magSANS_cs[name]:
        print("  ", name1)

In [ ]:
magSANS_cs_basic_analytical = magSANS_cs["analytical"]["basic"]
for name, item in magSANS_cs_basic_analytical:
    print(name, type(item))

In [ ]:
I_chi = magSANS_cs_basic_analytical["I_chi"]
print(I_chi)

In [ ]:
magSANS_cs_derived_analytical = magSANS_cs["analytical"]["derived"]
for name, item in magSANS_cs_derived_analytical:
    print(name, type(item))

In [ ]:
I_sf = magSANS_cs_derived_analytical["I_sf"]
print(I_sf)

In [ ]:
qz_plot = np.concatenate([qz.q, qz.q[:1]], axis=0)
qy_plot = np.concatenate([qy.q, qy.q[:1]], axis=0)
I_sf_plot = np.concatenate([I_sf.q, I_sf.q[:1]], axis=0)

fig, ax = plt.subplots()
mesh = ax.pcolormesh(
    qz_plot,
    qy_plot,
    I_sf_plot,
    shading="gouraud",
)
fig.colorbar(mesh, ax=ax, label=r"$I_{\mathrm{sf}}$")
ax.set(xlabel=r"$q_zR$", ylabel=r"$q_yR$", aspect="equal")
plt.show()

In [ ]:
qz_plot = np.concatenate([qz.q, qz.q[:1]], axis=0)
qy_plot = np.concatenate([qy.q, qy.q[:1]], axis=0)
I_chi_plot = np.concatenate([I_chi.q, I_chi.q[:1]], axis=0)

fig, ax = plt.subplots()
mesh = ax.pcolormesh(
    qz_plot,
    qy_plot,
    I_chi_plot,
    shading="gouraud",
)
fig.colorbar(mesh, ax=ax, label=r"$I_{\chi}$")
ax.set(xlabel=r"$q_zR$", ylabel=r"$q_yR$", aspect="equal")
plt.show()